# 01 . Feasibility check before committing

MEG-Mamba's tokeniser was trained on recordings beamformed directly to Schaefer100 at
250 Hz. We are feeding it a remap of someone else's Brainnetome reconstruction, which is
a different operation. If the tokeniser does not recognise the input, every downstream
number would be meaningless, and it could look like an honest null.

This notebook tokenises a handful of subjects and asks whether the tokens look like
real signal or like noise. Cheap check before the full run, same logic as running frozen
features before a long fine-tune.

**Prerequisites**

```
git clone https://github.com/OHBA-analysis/EphysTokenizer.git
```

EphysTokenizer needs its own environment (`etkn` in their docs). If it will not install,
stop here and raise it, because nothing downstream works without it.

In [11]:
import os, sys
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, h5py, scipy.signal

DATA   = '../data'
OUT    = '../results_mamba'
PARCEL = '../results_parcel'
MAMBA  = '/Users/faenegoro/Documents/Neuroimaging/MEG_Mamba/MEG-Mamba'

FS_ORIG, FS_TARGET = 600, 250      # MEG-Mamba wants 250 Hz, not the 256 BrainOmni uses
N_CHECK = 4                        # subjects to test

W    = np.load(f'{OUT}/schaefer_W_counts.npy')
keep = np.load(f'{PARCEL}/parcel_keep_idx_cortical.npy')
print(f'W {W.shape}  keep_idx {keep.shape}')
print(f'resampling {FS_ORIG} -> {FS_TARGET} Hz')

W (210, 100)  keep_idx (210,)
resampling 600 -> 250 Hz


In [12]:
def pca_merge(sig, W):
    out = np.zeros((W.shape[1], sig.shape[1]), np.float32)
    for p in range(W.shape[1]):
        m = np.where(W[:, p] > 0)[0]
        if len(m) == 0:
            continue
        X = sig[m] - sig[m].mean(1, keepdims=True)
        X = X / (X.std(1, keepdims=True) + 1e-12)
        U, Sv, Vt = np.linalg.svd(X, full_matrices=False)
        comp = Vt[0] * Sv[0]
        if np.corrcoef(comp, X.mean(0))[0, 1] < 0:
            comp = -comp
        out[p] = comp
    return out

def prepare(path, idx):
    """One subject -> (T, 100) at 250 Hz, the shape the tokeniser expects."""
    with h5py.File(path, 'r') as f:
        sig = np.array(f['matrix'][:, :, idx]).T[keep]        # (210, 36000) @ 600 Hz
    sig = pca_merge(sig, W)                                   # (100, 36000)
    sig = scipy.signal.resample_poly(sig, up=5, down=12, axis=1)   # 600 * 5/12 = 250 Hz
    return sig.T.astype(np.float32)                           # (T, 100)

x = prepare(f'{DATA}/ts48/tsAD.mat', 0)
print(f'prepared {x.shape}  expect (15000, 100) for 60 s at 250 Hz')
print(f'finite {np.isfinite(x).all()}  range {x.min():.2f} to {x.max():.2f}')

prepared (15000, 100)  expect (15000, 100) for 60 s at 250 Hz
finite True  range -15.36 to 17.44


In [13]:
# Load the bundled tokeniser. Needs EphysTokenizer importable.
sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/EphysTokenizer')
try:
    from ephys_tokenizer.models.ephys_tokenizer import EphysTokenizerModule
except ImportError as e:
    raise SystemExit(
        f'EphysTokenizer not importable ({e}).\n'
        'Clone it next to the other repos and install into its own env:\n'
        '  git clone https://github.com/OHBA-analysis/EphysTokenizer.git')

tok = EphysTokenizerModule.load_model(f'{MAMBA}/tokenizer/model')
print('tokeniser loaded')

tokeniser loaded


In [14]:
# Tokenise a few subjects and look at the distribution. The question is whether the
# tokeniser is using its vocabulary or collapsing everything onto a few codes.
import collections
VOCAB = 92     # from model_config.json

rows = []
for name, path in [('AD', f'{DATA}/ts48/tsAD.mat'), ('CONT', f'{DATA}/ts48/tsCONT.mat')]:
    for i in range(N_CHECK):
        t = tok.tokenize_session(prepare(path, i))       # (T, 100) uint8
        used = len(np.unique(t))
        counts = np.bincount(np.asarray(t).ravel(), minlength=VOCAB).astype(float)
        p = counts / counts.sum()
        ent = -(p[p > 0] * np.log2(p[p > 0])).sum()      # bits, max is log2(VOCAB)
        top = p.max()
        rows.append((name, i, t.shape, used, ent, top))
        print(f'  {name} subj {i}: tokens {t.shape}  distinct {used}/{VOCAB}  '
              f'entropy {ent:.2f}/{np.log2(VOCAB):.2f} bits  most common token {100*top:.1f}%')

  AD subj 0: tokens (15000, 100)  distinct 85/92  entropy 4.83/6.52 bits  most common token 10.8%
  AD subj 1: tokens (15000, 100)  distinct 84/92  entropy 4.81/6.52 bits  most common token 11.2%
  AD subj 2: tokens (15000, 100)  distinct 87/92  entropy 4.82/6.52 bits  most common token 10.9%
  AD subj 3: tokens (15000, 100)  distinct 85/92  entropy 4.80/6.52 bits  most common token 10.6%
  CONT subj 0: tokens (15000, 100)  distinct 83/92  entropy 4.85/6.52 bits  most common token 10.9%
  CONT subj 1: tokens (15000, 100)  distinct 85/92  entropy 4.84/6.52 bits  most common token 11.1%
  CONT subj 2: tokens (15000, 100)  distinct 83/92  entropy 4.84/6.52 bits  most common token 11.3%
  CONT subj 3: tokens (15000, 100)  distinct 87/92  entropy 4.90/6.52 bits  most common token 9.3%


## How to read the numbers above

**distinct tokens.** If only a handful of the 92 codes appear, the tokeniser is not
representing the signal, it is collapsing it.

**entropy.** Maximum is log2(92), about 6.5 bits. Healthy real signal should use a good
part of that range. Very low entropy means the tokens carry almost no information.

**most common token.** If one code takes a large share, the input probably looks like
noise or a constant to the tokeniser.

There is no published threshold to compare against, so this is a smell test rather than a
pass or fail. The useful comparison is the next cell, tokenising white noise through the
same path. If our data scores like noise, the remapping did not work and we stop.

In [15]:
# Control: pure noise through the identical path. If real data and noise look the same,
# the tokeniser is not seeing structure in our input.
rng = np.random.default_rng(123)
noise = rng.standard_normal((x.shape[0], x.shape[1])).astype(np.float32)
tn = tok.tokenize_session(noise)
cn = np.bincount(np.asarray(tn).ravel(), minlength=VOCAB).astype(float); pn = cn / cn.sum()
ent_n = -(pn[pn > 0] * np.log2(pn[pn > 0])).sum()

ent_real = np.mean([r[4] for r in rows])
print(f'real data  mean entropy {ent_real:.2f} bits   distinct {np.mean([r[3] for r in rows]):.0f}/{VOCAB}')
print(f'white noise     entropy {ent_n:.2f} bits   distinct {len(np.unique(tn))}/{VOCAB}')
print()
if ent_real < ent_n * 1.05:
    print('WARNING. Our data tokenises about as structurelessly as noise.')
    print('The Brainnetome to Schaefer remap probably does not match what the tokeniser')
    print('was trained on. Do not build the rest of the pipeline on this.')
else:
    print('Real data is more structured than noise. Worth continuing to notebook 02.')

real data  mean entropy 4.84 bits   distinct 85/92
white noise     entropy 3.57 bits   distinct 73/92

Real data is more structured than noise. Worth continuing to notebook 02.
